In [171]:
import pandas as pd
from pycountry import countries
import json
import datetime

In [172]:
BASE_URL = 'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_19-covid-{STATE}.csv'

In [173]:
def get_csse_iso_map():
    with open('csse_map_v2.json', "r") as of:
        return json.load(of)

In [174]:
def read_df(suffix):
    return (
        pd
        .read_csv(BASE_URL.format(STATE=suffix))
        .assign(
            code=lambda x: x['Country/Region'].map(get_csse_iso_map()),
        )
        .loc[lambda x: x["code"].notna().index]
        .drop(columns=["Province/State", "Lat", "Long", 'Country/Region'])
        .groupby(by="code", as_index=False)
        .sum()
        .assign(compartment=suffix.lower())
        .set_index(["code", "compartment"])
        .pipe(lambda x: x.set_axis(x.columns.map(pd.to_datetime), axis=1))
        .astype(int)
    )

In [176]:
_dfs = []
for state in ["Confirmed", "Deaths", "Recovered"]:
    _dfs.append(read_df(state))

dfs = pd.concat(_dfs)

In [177]:
dfs

,,2020-01-22,2020-01-23,2020-01-24,2020-01-25,2020-01-26,2020-01-27,2020-01-28,2020-01-29,2020-01-30,2020-01-31,...,2020-03-13,2020-03-14,2020-03-15,2020-03-16,2020-03-17,2020-03-18,2020-03-19,2020-03-20,2020-03-21,2020-03-22
code,compartment,,,,,,,,,,,,,,,,,,,,,
AFG,confirmed,0,0,0,0,0,0,0,0,0,0,...,7,11,16,21,22,22,22,24,24,40
AGO,confirmed,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,2,2
ALB,confirmed,0,0,0,0,0,0,0,0,0,0,...,33,38,42,51,55,59,64,70,76,89
AND,confirmed,0,0,0,0,0,0,0,0,0,0,...,1,1,1,2,39,39,53,75,88,113
ARE,confirmed,0,0,0,0,0,0,0,4,4,4,...,85,85,98,98,98,113,140,140,153,153
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
VNM,recovered,0,0,0,0,0,0,0,0,0,0,...,16,16,16,16,16,16,16,16,17,17
XKX,recovered,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ZAF,recovered,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Ours

In [206]:
ft = (
    pd
    .read_csv('../a.csv')
    .pipe(lambda x: x[(x["kind"] == "country") & x.est_active.notna()])
    .filter(items=["name", "est_active", "pop"])
    .assign(
        code=lambda x: x['name'].map(get_csse_iso_map()),
        est_active=lambda x: x["est_active"].astype(int),
    )
    .drop(columns=["name"])
    .dropna(subset=["code"])
    .set_index(["code"])
)

In [329]:
def serialize_country(csse_c, ft_c):
    res = {}
    for col_date, data in csse_c.items():
        data.index = data.index.map({
            'confirmed': "JH_Confirmed", 'deaths': "JH_Deaths", 'recovered': "JH_Recovered"}
        )
        res[col_date.strftime(format="%Y-%m-%d")] = {**data.to_dict(), "FT_Infected": ft_c["est_active"]}
    return {"estimates": {"days": res}}

In [330]:
#serialize_country(dfs.loc["USA"], ft.loc["USA"])

In [331]:
beds = (
    pd
    .read_csv('beds.csv')
    .assign(code=lambda x: x["country"].map(get_csse_iso_map()))
    .dropna(subset=["code"])
    .set_index('code')
)

In [332]:
import yaml

In [333]:
with open('../config-local.yaml', "r") as cfgi:
    region_mappings = yaml.load(cfgi, Loader=yaml.Loader)["regions_iso_mapping"]

In [338]:
res = {"created": datetime.datetime.now().isoformat().split('.')[0], 
       "created_by": "daniel-hnyk",
}

regions = {}
for iso_country_code in region_mappings.keys():
    ft_c = ft.loc[iso_country_code]
    if iso_country_code is None:
        breakpoint()
        
    try:
        csse_c = dfs.loc[iso_country_code]
    except KeyError:
        print(iso_country_code, "not present in csse data")
        continue
        
    
    name = countries.get(alpha_3=iso_country_code)
    if not name:
        print("not present name", iso_country_code)
        continue
        
    if iso_country_code in beds.index:
        cap_1000p = beds.loc[iso_country_code, 'cap/1000p']
        beds_p_100k = beds.loc[iso_country_code, 'beds/100k']
    else:
        print("Missing beds for", iso_country_code)
        cap_1000p = None
        beds_p_100k = None
        
    country_data = {
           "name": name.name,
           "kind": "state",
           "iso_alpha_3": iso_country_code,
           "population": ft_c["pop"],
           "capacity_1000p": cap_1000p,
           "beds_p_100k": beds_p_100k, 
    }
    country_data["data"] = serialize_country(csse_c, ft_c)
    regions[region_mappings[iso_country_code]] = country_data
    
res["regions"] = regions

Missing beds for AUS
Missing beds for RUS


In [335]:
import numpy

In [336]:
class NumpyEncode(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, numpy.integer):
            return int(obj)
        elif isinstance(obj, numpy.floating):
            return float(obj)
        elif isinstance(obj, numpy.ndarray):
            return obj.tolist()
        else:
            return super(NumpyEncode, self).default(obj)

In [337]:
with open('case-data-readable.json', "w") as ofile:
    json.dump(res, ofile, cls=NumpyEncode, indent=2)

In [341]:
## For vojta

In [359]:
(ft
 .loc[ft.index.isin(region_mappings.keys())]
 .assign(
     name=lambda x: x.index.map(lambda y: countries.get(alpha_3=y).name),
     risk=lambda x: x["est_active"]/x["pop"],
 )
).to_csv("case-map-data.csv", float_format='%.15f')